# 🚀 Step 2 — Run Qwen3.6 LLM Server on Kaggle GPU

This notebook launches a **fully functional LLM server** on Kaggle's free GPU and exposes it to the internet via a Cloudflare tunnel — no paid plan, no local GPU required.

### Architecture

```
┌─────────────────────────────────────────────────────┐
│                  Kaggle GPU Server                  │
│                                                     │
│   llama-server (Qwen3.6 + mmproj)                  │
│        ↕  localhost:8080                            │
│   cloudflared tunnel                                │
│        ↕  https://xxxx.trycloudflare.com            │
└─────────────────────────────────────────────────────┘
               ↕  internet
┌─────────────────────────────────────────────────────┐
│  Your local machine (any OpenAI-compatible client)  │
│  → chat.py, Open WebUI, Chatbox, Python SDK...     │
└─────────────────────────────────────────────────────┘
```

### Why these tools?

| Tool | Role | Why not the alternative? |
|---|---|---|
| `llama.cpp` (native binary) | LLM inference | Full CUDA control, `--mmproj` multimodal, no Python overhead |
| `Cloudflare Tunnel` | Public URL | Ngrok has strict request limits and cuts long token streams |
| `Kaggle` | Free GPU host | Colab disconnects after ~1.5h; Kaggle runs 12h headless |

---

### ⚠️ Prerequisites — Check before running

- [ ] **Accelerator** → **GPU T4 x2** for best performance, or T4 x1 (slower but works)
- [ ] **Internet** → **ON**
- [ ] **Dataset 1** → `qwen36-35b-a3b-gguf` attached under *Input*
- [ ] **Dataset 2** → `llama-cpp-bin` attached under *Input*

## ⚙️ Cell 1 — Configuration

**All parameters are here.** You should not need to change anything else in this notebook.

> 💡 **Thinking mode is not set here.** Control it per-conversation from your terminal:
> ```bash
> python chat.py --url https://xxxx.trycloudflare.com              # direct answers
> python chat.py --url https://xxxx.trycloudflare.com --thinking   # step-by-step reasoning
> ```
> No need to restart the server — switching modes is instant.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Edit this cell only
# ══════════════════════════════════════════════════════════════════════════════

# ── Your Kaggle username ──────────────────────────────────────────────────────
# Find it at: https://www.kaggle.com/settings (under 'Username')
KAGGLE_USERNAME = ""   # ← e.g. "johndoe"

# ── Model dataset (created in 1_setup_dataset.ipynb) ─────────────────────────
DATASET_NAME = "qwen36-35b-a3b-gguf"
MODEL_FILE   = "Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf"
MMPROJ_FILE  = "mmproj-F16.gguf"

# ── llama.cpp binaries dataset (created in 1b_setup_llama.ipynb) ─────────────
LLAMA_DATASET = "llama-cpp-bin"

# ── Server settings ───────────────────────────────────────────────────────────
PORT     = 8080
CTX_SIZE = 16384   # Context window in tokens (max 262144, more = more VRAM)

# ── Inference parameters (Unsloth official recommendations for Qwen3.6) ───────
# These are the non-thinking defaults.
# chat.py automatically switches to thinking params when --thinking is used.
TEMPERATURE      = 0.7
TOP_P            = 0.8
TOP_K            = 20
MIN_P            = 0.0
PRESENCE_PENALTY = 1.5

# ── GPU offloading ────────────────────────────────────────────────────────────
# 999 = as many layers as possible on GPU (auto-stops at VRAM limit)
# 1x T4 (15 GB): partial GPU + RAM → works but slower
# 2x T4 (30 GB): entire model on GPU → full speed
N_GPU_LAYERS = 999

# ══════════════════════════════════════════════════════════════════════════════

import os
from pathlib import Path

if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
if not KAGGLE_USERNAME:
    raise ValueError("Please set KAGGLE_USERNAME at the top of this cell.")

# Dataset paths
DATASET_BASE  = Path(f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{DATASET_NAME}")
LLAMA_BIN_DIR = Path(f"/kaggle/input/datasets/{KAGGLE_USERNAME}/{LLAMA_DATASET}")
MODEL_PATH    = DATASET_BASE / MODEL_FILE
MMPROJ_PATH   = DATASET_BASE / MMPROJ_FILE

# Binaries will be copied here (writable, .so files in the same dir)
WORKING       = Path("/kaggle/working")
LLAMA_SERVER  = WORKING / "llama-server"

assert MODEL_PATH.exists(),    f"❌ Model not found: {MODEL_PATH}\n   Did you attach the qwen36-35b-a3b-gguf dataset?"
assert MMPROJ_PATH.exists(),   f"❌ mmproj not found: {MMPROJ_PATH}"
assert LLAMA_BIN_DIR.exists(), f"❌ llama-cpp-bin dataset not found: {LLAMA_BIN_DIR}\n   Did you attach the llama-cpp-bin dataset?"

print("✅ Configuration validated")
print(f"   Model        : {MODEL_PATH}")
print(f"   mmproj       : {MMPROJ_PATH}")
print(f"   llama-cpp    : {LLAMA_BIN_DIR}")
print(f"   Context size : {CTX_SIZE} tokens")

## 🖥️ Cell 2 — GPU check

In [ ]:
import subprocess

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True
)

if result.returncode != 0:
    print("❌ No GPU detected. Go to Settings → Accelerator and select T4 GPU.")
else:
    lines = result.stdout.strip().split("\n")
    total_vram = 0
    print(f"{'GPU':<5} {'Name':<20} {'Total VRAM':>12} {'Free VRAM':>12}")
    print("-" * 55)
    for i, line in enumerate(lines):
        name, total, free = [x.strip() for x in line.split(",")]
        total_mb = int(total.replace(" MiB", ""))
        free_mb  = int(free.replace(" MiB", ""))
        total_vram += total_mb
        print(f"  {i:<3} {name:<20} {total_mb/1024:>10.1f} GB {free_mb/1024:>10.1f} GB")
    print("-" * 55)
    print(f"  Total available VRAM: {total_vram/1024:.1f} GB")
    print()
    model_size_gb = MODEL_PATH.stat().st_size / (1024**3)
    if total_vram / 1024 >= model_size_gb:
        print(f"✅ Enough VRAM ({total_vram/1024:.1f} GB) for the full model ({model_size_gb:.1f} GB) on GPU.")
    else:
        print(f"⚠️  VRAM ({total_vram/1024:.1f} GB) < model size ({model_size_gb:.1f} GB).")
        print(f"   Hybrid mode: GPU layers + RAM overflow. Works, but slower.")
        print(f"   → For full speed: switch to 2x T4 in Kaggle accelerator settings.")

## 📦 Cell 3 — Load llama.cpp binaries from dataset

We copy the pre-compiled binaries **and** their shared libraries (`.so` files) from the
`llama-cpp-bin` dataset into `/kaggle/working/` (writable).

Then we set `LD_LIBRARY_PATH` so the OS finds the `.so` files at runtime.
Without this, `llama-server` crashes immediately with `cannot open shared object file`.

> ⏳ Expected time: **~5 seconds** (vs ~26 minutes of compilation).

In [ ]:
import shutil, json, os, subprocess
from pathlib import Path

# ── Copy binaries ─────────────────────────────────────────────────────────────
binaries = ["llama-server", "llama-cli", "llama-mtmd-cli"]
for name in binaries:
    src = LLAMA_BIN_DIR / name
    dst = WORKING / name
    assert src.exists(), (
        f"❌ Binary '{name}' not found in dataset.\n"
        f"   Re-run 1b_setup_llama.ipynb to rebuild the dataset."
    )
    shutil.copy2(src, dst)
    dst.chmod(0o755)

# ── Copy shared libraries (.so) ───────────────────────────────────────────────
# llama-server is dynamically linked — it needs its .so files at runtime.
# We copy them to /kaggle/working/ alongside the binaries.
so_count = 0
for so_file in LLAMA_BIN_DIR.glob("*.so*"):
    shutil.copy2(so_file, WORKING / so_file.name)
    so_count += 1

# ── Set LD_LIBRARY_PATH ───────────────────────────────────────────────────────
# Tell the OS where to find the .so files when llama-server starts.
ld_path = f"/kaggle/working:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ["LD_LIBRARY_PATH"] = ld_path

# ── Verify binary works ───────────────────────────────────────────────────────
test = subprocess.run(
    [str(LLAMA_SERVER), "--version"],
    capture_output=True, text=True,
    env={**os.environ, "LD_LIBRARY_PATH": ld_path}
)

if test.returncode != 0:
    print("❌ llama-server failed sanity check:")
    print(test.stderr[:500])
    raise RuntimeError("Binary verification failed — check dataset contents.")

# ── Print summary ─────────────────────────────────────────────────────────────
ver_file = LLAMA_BIN_DIR / "version.json"
if ver_file.exists():
    info = json.loads(ver_file.read_text())
    print("✅ llama.cpp binaries loaded.")
    print(f"   Commit  : {info.get('llama_cpp_commit')}  ({info.get('commit_date')})")
    print(f"   Version : {info.get('llama_server_ver')}")
    print(f"   CUDA    : {info.get('cuda_arch')}")
else:
    ver_str = (test.stdout or test.stderr).strip().splitlines()[0]
    print(f"✅ llama.cpp binaries loaded: {ver_str}")

print(f"   Copied  : {len(binaries)} binaries + {so_count} shared libraries")
print(f"   LD path : {ld_path.split(':')[0]}")

## ☁️ Cell 4 — Install Cloudflare Tunnel

`cloudflared` creates a secure public URL forwarding traffic to our local server.

Why Cloudflare over Ngrok?
- **No account or token needed** — works instantly in anonymous mode
- **No request limits** — Ngrok's free tier blocks the API after too many calls
- **No streaming timeout** — Ngrok cuts long HTTP connections, breaking LLM token streaming

In [ ]:
%%bash
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
     -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared
echo "✅ cloudflared installed: $(cloudflared --version)"

## 🧠 Cell 5 — Launch llama-server

Starts `llama-server` in the background, then waits until `/health` returns `{"status": "ok"}`
before continuing. Checking for `ok` (not just HTTP 200) ensures the model is fully loaded.

**Thinking mode** is not set here — it is passed per request by `chat.py`.
No server restart needed to switch modes.

In [ ]:
import subprocess, time, requests, os

# Kill any leftover llama-server from a previous run
subprocess.run(["pkill", "-f", "llama-server"], capture_output=True)
time.sleep(2)

# LD_LIBRARY_PATH must be in the environment of the subprocess too
env = {**os.environ, "LD_LIBRARY_PATH": f"/kaggle/working:{os.environ.get('LD_LIBRARY_PATH', '')}"}

cmd = [
    str(LLAMA_SERVER),
    "--model",            str(MODEL_PATH),
    "--mmproj",           str(MMPROJ_PATH),      # vision projector — enables image input
    "--alias",            "qwen3.6-35b-a3b",
    "--host",             "0.0.0.0",
    "--port",             str(PORT),
    "--n-gpu-layers",     str(N_GPU_LAYERS),
    "--ctx-size",         str(CTX_SIZE),
    "--temp",             str(TEMPERATURE),
    "--top-p",            str(TOP_P),
    "--top-k",            str(TOP_K),
    "--min-p",            str(MIN_P),
    "--presence-penalty", str(PRESENCE_PENALTY),
    "--log-file",         "/tmp/llama-server.log",
    # Thinking mode is NOT set here — passed per request by chat.py
]

print("🚀 Starting llama-server...")
print(f"   Model  : {MODEL_FILE}")
print(f"   mmproj : {MMPROJ_FILE}")
print(f"   💡 Thinking: use chat.py --thinking  (no restart needed)")
print()

server_proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env)

print("⏳ Loading model into GPU memory (this may take 5–6 minutes)...")

# Wait for {"status": "ok"} — not just HTTP 200
# llama-server returns 200 + {"status":"loading"} while loading,
# and 200 + {"status":"ok"} only when truly ready to serve requests.
timeout, interval, elapsed, server_ok = 600, 15, 0, False
while elapsed < timeout:
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=2)
        if r.status_code == 200 and r.json().get("status") == "ok":
            server_ok = True
            break
    except Exception:
        pass
    print(f"   [{elapsed:>3}s] Still loading...", flush=True)
    time.sleep(interval)
    elapsed += interval

if not server_ok:
    print("\n❌ Server failed to start. Last log lines:")
    try:
        with open("/tmp/llama-server.log") as f:
            print("".join(f.readlines()[-20:]))
    except Exception:
        print("   (log file not found — binary may have crashed before writing logs)")
    raise RuntimeError("Server did not become healthy in time.")

print(f"\n✅ Server is ready! (started in {elapsed}s)")
print(f"   Local endpoint: http://localhost:{PORT}/v1")

## 🌐 Cell 6 — Start Cloudflare Tunnel & get public URL

In [ ]:
import re

tunnel_log = "/tmp/cloudflare.log"
tunnel_url = None

with open(tunnel_log, "w") as log:
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
        stdout=log, stderr=log,
    )

print("⏳ Waiting for Cloudflare tunnel URL...")

for _ in range(30):
    time.sleep(2)
    try:
        with open(tunnel_log) as f:
            match = re.search(r"https://[\w-]+\.trycloudflare\.com", f.read())
        if match:
            tunnel_url = match.group(0)
            break
    except Exception:
        pass

if not tunnel_url:
    print("❌ Could not get Cloudflare URL. Last log:")
    with open(tunnel_log) as f:
        print(f.read()[-1000:])
    raise RuntimeError("Cloudflare tunnel failed to start.")

print()
print("═" * 60)
print(f"  🔗 YOUR SERVER URL")
print(f"  {tunnel_url}")
print("═" * 60)
print()
print(f"  OpenAI endpoint : {tunnel_url}/v1")
print(f"  Direct answers  : python chat.py --url {tunnel_url}")
print(f"  Thinking mode   : python chat.py --url {tunnel_url} --thinking")

## 🔁 Cell 7 — Keep-alive loop

Keeps the notebook alive for up to 12 hours. Prints a health check every 5 minutes.

**To stop:** interrupt the kernel (■ button) or let the session expire.

In [ ]:
import datetime

start_time     = datetime.datetime.now()
check_interval = 300

print(f"🟢 Server running since {start_time.strftime('%H:%M:%S')}")
print(f"   URL     : {tunnel_url}")
print(f"   Model   : {MODEL_FILE}")
print(f"   Timeout : ~12h (Kaggle session limit)")
print()
print(f"   Connect : python chat.py --url {tunnel_url}")
print(f"   Thinking: python chat.py --url {tunnel_url} --thinking")
print()
print("   Press ■ (interrupt kernel) to stop.")
print("─" * 60)

try:
    while True:
        time.sleep(check_interval)
        elapsed = datetime.datetime.now() - start_time
        h, rem  = divmod(int(elapsed.total_seconds()), 3600)
        m       = rem // 60
        try:
            r      = requests.get(f"http://localhost:{PORT}/health", timeout=5)
            status = "✅ healthy" if r.status_code == 200 else f"⚠️ {r.status_code}"
        except Exception:
            status = "❌ unreachable"
        ts = datetime.datetime.now().strftime('%H:%M:%S')
        print(f"  [{ts}]  Uptime: {h}h {m:02d}m  |  {status}")
        if "❌" in status:
            print("\n⚠️  Server crashed. Check /tmp/llama-server.log")
            break
except KeyboardInterrupt:
    print("\n🛑 Shutting down...")
    server_proc.terminate()
    tunnel_proc.terminate()
    print("   Done.")